# 第 4 课：MCP 与安全文件服务

预计用时：60–90 分钟  
适合人群：完成上一课的零基础学习者；本 Notebook 也包含独立运行所需的准备代码。

## 学习目标

- 理解 MCP 解决的连接标准化问题
- 用 FastMCP 暴露工具
- 防御路径穿越、超大文件与危险扩展名

## 学习方式

按顺序运行每个代码单元格。先阅读预测结果，再运行验证；遇到报错先看本课“常见问题”，不要直接跳过。带有真实模型或外网请求的示例默认注释，确认 API Key 与费用后再启用。


## 先别急着看代码

这一课只做三件事：

1. 创建一个只能访问指定目录的文件服务
2. 尝试正常路径
3. 尝试越界路径并确认它被拒绝

第一次学习时，只要求能按顺序运行并用自己的话解释结果。类、类型注解和异常处理的全部细节，不需要一次记住。


## 本课术语卡

- **MCP**：让 AI 客户端用统一方式连接工具的协议
- **Server**：提供工具的一方
- **路径穿越**：用 `../` 等方式访问允许目录之外的文件
- **白名单**：只允许明确列出的类型

看到陌生英文时先回到这里。一个术语只需要先记住一句话。


## 推荐学习动作

每个代码格都按这个顺序学习：

1. 先读上方说明，只找“输入”和“输出”。
2. 不修改代码，按 `Shift + Enter` 运行。
3. 看实际结果是否符合说明。
4. 只改一个最小值，再运行一次。

如果报 `NameError`，通常是漏跑了前面的格子；选择 **Restart Kernel and Run All** 可以从头重来。


## 0. 最小热身：先理解允许目录

这一段与后面完整工程代码相互独立。先运行它，立刻看到结果。


In [ ]:
from pathlib import Path

allowed_root = (Path.cwd() / "agent_workspace" / "mcp_files").resolve()
normal_file = (allowed_root / "notes.txt").resolve()
outside_file = (allowed_root / ".." / "secret.txt").resolve()

print("正常文件在目录内：", allowed_root in normal_file.parents)
print("越界文件在目录内：", allowed_root in outside_file.parents)


**你应该观察到什么？**

后面的 `safe_path` 正是利用这个判断：第二行如果是 False，就拒绝访问。

如果结果符合说明，再继续下面的完整版本。


## 1. 先理解概念

MCP 让客户端以统一协议发现和调用外部能力。文件工具风险很高：仅检查字符串中有没有 `..` 不够，必须解析成绝对路径后确认目标仍在允许的根目录内。

### 本课路线

1. 创建专用 MCP 根目录
2. 实现并测试 `safe_path`
3. 添加读、写、列目录和搜索工具
4. 限制大小、扩展名和结果数量
5. 了解如何在独立进程启动服务


## 2. 运行前检查

1. 从项目根目录启动 Jupyter Lab。
2. 选择项目 `.venv` 对应的 Python 内核。
3. 若本课调用百炼，先在启动 Jupyter 的终端设置 `DASHSCOPE_API_KEY`。
4. 不要把 Key 粘贴到单元格、截图或 Git 提交中。

> 下方“准备代码”可能与前课重复，这是为了保证每个 Notebook 都能单独运行。初学时建议展开阅读，熟悉后可折叠。


### 准备代码


### 现在做什么？

这一小格代码

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
# %pip install -q openai pydantic>=2.7 httpx fastapi uvicorn fastmcp langgraph langfuse ragas numpy pytest

import os
from dotenv import load_dotenv

load_dotenv()

# 推荐在启动 Jupyter 前设置：
# Windows PowerShell: $env:DASHSCOPE_API_KEY='sk-...'
# macOS/Linux:       export DASHSCOPE_API_KEY='sk-...'

BAILIAN_API_KEY = os.getenv('DASHSCOPE_API_KEY', '')
BAILIAN_BASE_URL = os.getenv(
    'BAILIAN_BASE_URL',
    'https://dashscope.aliyuncs.com/compatible-mode/v1',
)
BAILIAN_MODEL = os.getenv('BAILIAN_MODEL', 'qwen-plus')
BAILIAN_EMBEDDING_MODEL = os.getenv('BAILIAN_EMBEDDING_MODEL', 'text-embedding-v4')

print('模型:', BAILIAN_MODEL)
print('Base URL:', BAILIAN_BASE_URL)
print('API Key:', '已配置' if BAILIAN_API_KEY else '未配置（调用模型前必须设置）')


### 准备代码


### 现在做什么？

这一小格代码

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
from __future__ import annotations

import asyncio
import json
import logging
import math
import sqlite3
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Awaitable, Callable, Literal, TypedDict

import httpx
import numpy as np
from openai import AsyncOpenAI
from pydantic import BaseModel, ConfigDict, Field, ValidationError

WORKSPACE = (Path.cwd() / 'agent_workspace').resolve()
WORKSPACE.mkdir(exist_ok=True)

def require_api_key() -> None:
    if not BAILIAN_API_KEY:
        raise RuntimeError('请先设置环境变量 DASHSCOPE_API_KEY，然后重新运行配置单元格。')

client = AsyncOpenAI(api_key=BAILIAN_API_KEY or 'missing', base_url=BAILIAN_BASE_URL)
print('工作目录:', WORKSPACE)


### 核心实验


### 现在做什么？

先准备变量和依赖

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
from fastmcp import FastMCP

MCP_ROOT = (WORKSPACE / 'mcp_files').resolve()
MCP_ROOT.mkdir(exist_ok=True)
MAX_FILE_BYTES = 1_000_000
ALLOWED_WRITE_SUFFIXES = {'.txt', '.md', '.json', '.csv', '.py'}


### 现在做什么？

安全路径函数是本课最重要的一格：解析真实路径后，再确认它没有跑出允许目录。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
def safe_path(relative_path: str) -> Path:
    if Path(relative_path).is_absolute():
        raise ValueError('不允许绝对路径')
    target = (MCP_ROOT / relative_path).resolve()
    if target != MCP_ROOT and MCP_ROOT not in target.parents:
        raise ValueError('路径越过工作目录')
    return target


### 现在做什么？

先准备变量和依赖

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
mcp = FastMCP('safe-filesystem')


### 现在做什么？

定义 `read_file`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
@mcp.tool
def read_file(path: str) -> str:
    target = safe_path(path)
    if not target.is_file():
        raise FileNotFoundError(path)
    if target.stat().st_size > MAX_FILE_BYTES:
        raise ValueError('文件过大')
    return target.read_text(encoding='utf-8')


### 现在做什么？

定义 `write_file`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
@mcp.tool
def write_file(path: str, content: str, overwrite: bool = False) -> dict[str, Any]:
    target = safe_path(path)
    if target.suffix.lower() not in ALLOWED_WRITE_SUFFIXES:
        raise ValueError('不允许写入该扩展名')
    if len(content.encode('utf-8')) > MAX_FILE_BYTES:
        raise ValueError('内容过大')
    if target.exists() and not overwrite:
        raise FileExistsError('文件已存在；若确定覆盖，请设置 overwrite=true')
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
    return {'path': str(target.relative_to(MCP_ROOT)), 'bytes': target.stat().st_size}


### 现在做什么？

定义 `list_directory`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
@mcp.tool
def list_directory(path: str = '.') -> list[dict[str, Any]]:
    target = safe_path(path)
    if not target.is_dir():
        raise NotADirectoryError(path)
    return [
        {'name': child.name, 'type': 'directory' if child.is_dir() else 'file', 'size': child.stat().st_size if child.is_file() else None}
        for child in sorted(target.iterdir())[:200]
    ]


### 现在做什么？

定义 `search_content`：先看输入，再看返回值，函数内部细节可以第二遍再读。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
@mcp.tool
def search_content(query: str, path: str = '.', max_results: int = 50) -> list[dict[str, Any]]:
    if not query or len(query) > 200:
        raise ValueError('query 长度必须为 1-200')
    target = safe_path(path)
    results: list[dict[str, Any]] = []
    for file in target.rglob('*'):
        if len(results) >= min(max_results, 100):
            break
        if not file.is_file() or file.suffix.lower() not in ALLOWED_WRITE_SUFFIXES or file.stat().st_size > MAX_FILE_BYTES:
            continue
        for line_no, line in enumerate(file.read_text(encoding='utf-8', errors='replace').splitlines(), 1):
            if query.casefold() in line.casefold():
                results.append({'path': str(file.relative_to(MCP_ROOT)), 'line': line_no, 'text': line[:300]})
                if len(results) >= min(max_results, 100):
                    break
    return results


### 现在做什么？

最后把上面的零件连接起来并做一次检查。

运行前不用逐行背诵。先猜测：这一格会“只定义东西”，还是会“立刻打印或写入结果”？


In [ ]:
assert safe_path('notes/demo.md').is_relative_to(MCP_ROOT)
try:
    safe_path('../../secret.txt')
    raise AssertionError('路径穿越测试本应失败')
except ValueError:
    print('路径穿越已正确阻止')

# 在普通 Python 文件中启动：mcp.run(transport='streamable-http', host='0.0.0.0', port=8001)


## 3. 观察与验证

核心代码中的真实 API 调用默认被注释。先运行无需额度的断言或定义单元格；确认输出和预期一致后，再逐行取消示例注释。


## 4. 代码讲解

Notebook 负责定义与本地测试服务，不直接阻塞运行 `mcp.run()`。生产环境还必须处理符号链接、认证、租户隔离、并发写入和审计。

调试建议：从报错的最后一行开始读，确认当前 Notebook 的单元格是否按顺序全部运行；若看到 `NameError`，通常是准备单元格未运行或内核已重启。


## 5. 常见问题

- **`ModuleNotFoundError`**：确认选中了 `.venv` 内核，并重新安装 `requirements.txt`。
- **提示未配置 API Key**：在启动 Jupyter 的同一个终端设置环境变量，然后重启内核。
- **网络超时或 429**：公开接口或模型服务可能限流；稍后重试，不要移除超时保护。
- **运行结果和预期不同**：先执行“Restart Kernel and Run All”，排除旧变量残留。
- **产生费用吗？**：只有实际调用百炼聊天或 Embedding 接口才会消耗额度；本地定义、SQLite 和断言不会。

## 6. 练习

- 尝试 `../secret.txt` 并确认被拒绝
- 写入一个 `.exe` 文件并确认被拒绝
- 把 Server 代码迁移到独立 `mcp_server.py`

建议先复制相关单元格再修改，保留一份能工作的基线。


## 7. 本课验收

完成后逐项确认：

- [ ] 合法相对路径可读写
- [ ] 路径穿越测试通过
- [ ] 知道 Notebook 为什么不直接启动常驻服务

如果某项还解释不清，回到对应代码，用更小的输入单独调用函数，而不是直接运行完整 Agent。


## 下一步

继续学习 `05_LangGraph工作流.ipynb`。

> 学习记录建议：写下今天最重要的一个概念、遇到的一个错误、以及你如何验证修复。


## 一句话回顾

文件工具首先要限制边界，然后才考虑功能是否丰富。

### 如果你仍然觉得难

先只完成以下最低目标：

- 能从上到下运行本课；
- 能指出哪一格是输入、哪一格产生输出；
- 能用一句话说出本课解决了什么问题。

做到这三点就可以进入下一课。第二遍学习时再研究类型注解、异常分支和工程细节。
